In [ ]:
# Compare the saved pension article with its current Silver row without writing data.
from parse_article import parse_article
from pyspark.sql import functions as F

table_name = '`news-analytics`.silver.articles'
article_url = 'https://kontrast.at/pension-frauen-weniger/'
silver = spark.table(table_name)
before_count = silver.count()
current = (silver.filter(F.col('canonical_url') == article_url)
    .select('canonical_url', 'source_path', 'body_text')
    .first())
if current is None:
    raise ValueError(f'Article not found in Silver: {article_url}')

# Parse the original HTML using the updated parser and preview the recovered text.
file_row = (spark.read.format('binaryFile').load(current.source_path)
    .select('path', 'content').first())
article = parse_article(file_row.content.decode('utf-8', errors='replace'))
if article is None or article['canonical_url'] != article_url or not article['body_text'].strip():
    raise ValueError(f'No matching article body found in {current.source_path}')
print('SOURCE FILE:', file_row.path)
print('SILVER BODY CHARS:', len(current.body_text or ''))
print('PARSED BODY CHARS:', len(article['body_text']))
print('PARSED BODY PREVIEW:', article['body_text'][:300])
print('ARTICLE COUNT BEFORE:', before_count)


In [ ]:
# Update body text and links for this URL and source file only if Silver has an empty body.
from pyspark.sql.types import StringType, StructField, StructType

candidate_schema = StructType([
    StructField('canonical_url', StringType()),
    StructField('source_path', StringType()),
    StructField('body_text', StringType()),
    StructField('links', silver.schema['links'].dataType),
])
candidate = spark.createDataFrame([{
    'canonical_url': article['canonical_url'],
    'source_path': current.source_path,
    'body_text': article['body_text'],
    'links': article['links'],
}], schema=candidate_schema)
candidate.createOrReplaceTempView('repair_candidate')
merge_result = spark.sql(f'''
    MERGE INTO {table_name} AS target
    USING repair_candidate AS source
    ON target.canonical_url = source.canonical_url
        AND target.source_path = source.source_path
    WHEN MATCHED AND TRIM(target.body_text) = '' AND TRIM(source.body_text) <> ''
        THEN UPDATE SET body_text = source.body_text, links = source.links
''')
print('MERGE RESULT:', merge_result.collect())


In [ ]:
# Verify that the repaired row has text and the table still has the same article count.
updated = (spark.table(table_name)
    .filter(F.col('canonical_url') == article_url)
    .select('canonical_url', 'source_path', 'body_text', 'links')
    .first())
after_count = spark.table(table_name).count()
print('ARTICLE COUNT:', before_count, '->', after_count)
print('REPAIRED URL:', updated.canonical_url)
print('REPAIRED SOURCE:', updated.source_path)
print('BODY CHARS:', len(updated.body_text or ''))
print('LINKS:', len(updated.links or []))
print('BODY PREVIEW:', (updated.body_text or '')[:300])
